## **Gradient Boosting for Regression and Classification**

### **Topic Roadmap**

**1. Prepare regression and classification datasets**

**2. Fit `GradientBoostingRegressor`**

**3. Fit `GradientBoostingClassifier`**

**4. Compare evaluation metrics**

**5. Key revision notes**

## **1. Regression Dataset**

Gradient boosting builds an additive model by fitting new trees to the negative gradient of the loss.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, classification_report

RANDOM_STATE = 42
diabetes = load_diabetes(as_frame=True)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    diabetes.data, diabetes.target, test_size=0.2, random_state=RANDOM_STATE
)

## **2. Gradient Boosting for Regression**

For regression, each tree approximates the negative gradient of the squared-error loss.

In [2]:
regressor = GradientBoostingRegressor(
    n_estimators=150, learning_rate=0.05, max_depth=2,
    loss="squared_error", random_state=RANDOM_STATE
)
regressor.fit(Xr_train, yr_train)
r_pred = regressor.predict(Xr_test)
print({"MAE": mean_absolute_error(yr_test, r_pred), "RMSE": mean_squared_error(yr_test, r_pred) ** 0.5, "R2": r2_score(yr_test, r_pred)})

{'MAE': 42.70303510698817, 'RMSE': 52.67479021837492, 'R2': 0.47630161304332974}


## **3. Gradient Boosting for Classification**

For classification, the model adds trees to improve the log-loss objective and returns class probabilities.

In [3]:
cancer = load_breast_cancer(as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.2, stratify=cancer.target, random_state=RANDOM_STATE
)
classifier = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, max_depth=2,
    loss="log_loss", random_state=RANDOM_STATE
)
classifier.fit(Xc_train, yc_train)
c_pred = classifier.predict(Xc_test)
print(f"Accuracy: {accuracy_score(yc_test, c_pred):.3f}")
print(classification_report(yc_test, c_pred, target_names=cancer.target_names))

Accuracy: 0.939
              precision    recall  f1-score   support

   malignant       0.93      0.90      0.92        42
      benign       0.95      0.96      0.95        72

    accuracy                           0.94       114
   macro avg       0.94      0.93      0.93       114
weighted avg       0.94      0.94      0.94       114



## **4. Training Diagnostics**

The staged prediction methods show how validation performance changes as trees are added.

In [4]:
staged_scores = [accuracy_score(yc_test, pred) for pred in classifier.staged_predict(Xc_test)]
pd.DataFrame({"iteration": np.arange(1, len(staged_scores) + 1), "accuracy": staged_scores}).tail()

,iteration,accuracy
145,146,0.938596
146,147,0.938596
147,148,0.938596
148,149,0.938596
149,150,0.938596


### **Key Revision Notes**

- Gradient boosting fits learners sequentially, unlike bagging’s parallel resampling.
- `learning_rate` and `n_estimators` trade off against each other.
- Shallow trees are common weak learners.
- Use validation curves or early stopping strategies to control overfitting.